# C8-embeddings — Session 2: Embedding Matrices and the Similarity Matrix

*One class session, roughly 85 minutes. Builds on Session 1 (tokens,
KeyedVectors, the float64 boundary cast, pairwise cosine) and leans on
F3-matrices (matrix product, matrices as stacked rows) and
F1-scientific-python (broadcasting, axis sums, `keepdims`, matplotlib).*

**This session:** stacking token vectors into one **embedding matrix**
with this course's fixed convention — *rows are tokens*, shape
$(N, 100)$; row normalization in the exam's broadcasting-only register
(no `np.linalg`, no loops); the unit-row stack `W`; cosine similarity
read straight off unit rows; the session's centerpiece — the
**similarity matrix** $S = W W^{\mathsf T}$, all pairwise cosines in one
matrix product, with its symmetry and unit diagonal verified in code; a
heatmap reading of $S$'s block structure; a fully worked numeric
normal-form MC; and the shape pitfalls that cost real points.

Try every checkpoint by hand first, then verify in code.
Answers are collected at the end of this notebook.

In [ ]:
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import matplotlib.pyplot as plt

## 1. From Words to a Matrix

**Motivation.**
Session 1 measured one word pair at a time.
Real questions come in batches — *rank all these words against each
other* — and batch geometry wants a matrix.

**Definition (embedding matrix, this course's convention).**
Given $N$ tokens, stack their vectors as the **rows** of a matrix
$V \in \mathbb{R}^{N \times 100}$: row $i$ *is* token $i$'s vector.
Rows are tokens, columns are embedding dimensions, shape $(N, 100)$ —
fixed for this unit and consumed as-is by later units.

`gensim` cooperates: indexing `kv` with a *list* of words returns
exactly this stack, one row per word in list order.
The float64 boundary cast wraps the lookup, same cell, as always.

In [ ]:
import gensim.downloader

kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["harbor", "boat", "sailor", "tide", "violin", "cello", "pepper", "honey"]

V = np.asarray(kv[WORDS], dtype=np.float64)   # rows are tokens: (N, 100)
print("V shape:", V.shape, "| dtype:", V.dtype)

# row i is exactly word i's vector, in list order
print("row 0 is 'harbor':",
      bool(np.allclose(V[0], np.asarray(kv["harbor"], dtype=np.float64))))
print("row 5 is 'cello' :",
      bool(np.allclose(V[5], np.asarray(kv["cello"], dtype=np.float64))))

Keep `WORDS` next to `V` forever: the matrix has no memory of which row
is which word — the *list* is the row index's meaning.
Row $i$ of every array we build from here on (norms, unit rows,
similarity rows) refers to `WORDS[i]`.

### Checkpoint 1

1. For 12 tokens under this convention, give the shape of the matrix
   and say what `V[7]` and `V[:, 7]` each mean.
2. Why must the words list and the matrix travel together — what
   question can `V` alone not answer?
3. A file stores vectors for the same 8 words but as *columns*
   (shape `(100, 8)`). Write the one expression that converts it to
   this course's convention.

## 2. Row Norms in the Broadcasting Register

**Motivation.**
Cosine needs every row divided by its own length, so first we need all
$N$ row norms — and we need them in the register the exam actually
grades.
Round 1's implementation tasks put it bluntly: **no `np.linalg`, no
loops** — zero points otherwise.
Broadcasting and axis sums are the whole toolkit, and they are enough.

**The two-line derivation.**
$V * V$ squares every entry (elementwise); summing each *row* needs
`sum(axis=1)`; the square root finishes the norms:
$$\lVert v_i \rVert = \sqrt{\textstyle\sum_k V_{ik}^2}.$$

One subtlety decides everything downstream.
`(V * V).sum(axis=1)` has shape $(N,)$ — a flat vector.
We want shape $(N, 1)$ instead, and the switch that does it:
**`keepdims=True` keeps the summed-over axis as a length-1 axis, so the
result stays a column and broadcasts across each row.**
First on a toy seeded matrix where every number is checkable, then on
the real one.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
M = rng.normal(0, 1, (3, 4))
print("M:\n", np.round(M, 3))

flat = np.sqrt((M * M).sum(axis=1))                  # shape (3,)  -- flat
col  = np.sqrt((M * M).sum(axis=1, keepdims=True))   # shape (3,1) -- column
print("without keepdims:", flat.shape, "->", np.round(flat, 4))
print("with keepdims   :", col.shape)
print(np.round(col, 4))

In [ ]:
norms = np.sqrt((V * V).sum(axis=1, keepdims=True))   # (8, 1)
print("norms shape:", norms.shape)
for w, n in zip(WORDS, norms.ravel()):
    print(f"  {w:8} {n:.4f}")

Eight lengths between $5.07$ and $6.14$ — and *not equal*: raw GloVe
rows come in assorted lengths (`boat` at $6.12$, `tide` at $5.07$).
That spread is precisely why raw dot products are a biased similarity
measure and why the next section divides it out.

Why insist on the $(8, 1)$ column?
Because the very next operation is `V / norms`: dividing $(8, 100)$ by
$(8, 1)$ broadcasts the division across each row — row $i$ divided by
its own norm.
Dividing by the flat $(8,)$ version instead is a *shape error* (NumPy
would try to line up $100$ columns with $8$ norms), which mercifully
fails loudly; Section 8 shows the quieter cousin that doesn't.

### Checkpoint 2

1. Without running: `A` has shape `(5, 3)`. Give the shapes of
   `(A * A).sum(axis=1)`, `(A * A).sum(axis=1, keepdims=True)`, and
   `(A * A).sum(axis=0, keepdims=True)`.
2. By hand: compute the row norms of
   $\begin{pmatrix} 3 & 4 \\ 5 & 12 \end{pmatrix}$
   and write the `(2, 1)` result.
3. State the two banned tools this register replaces, and what the
   penalty for using them is in the exam's grading note.

## 3. The Unit-Row Stack `W`

Dividing each row by its norm rescales every token vector to length 1
while preserving its direction — Session 1's cosine logic, applied to
all rows at once.
The result is this unit's central object, and its name is part of the
convention: **`W`, the unit-normalized embedding matrix**, same
$(N, 100)$ shape, every row a unit vector.

In [ ]:
W = V / norms                                   # (8,100) / (8,1) -> (8,100)
print("W shape:", W.shape)

row_lengths = np.sqrt((W * W).sum(axis=1))      # every entry should be 1.0
print("unit row norms:", np.round(row_lengths, 12))
assert np.allclose(row_lengths, 1.0)

All eight rows have length exactly $1$ (to machine precision — the
printed values are rounded to 12 places).

**A convention that outlives this unit.**
Everything downstream consumes `W` in exactly this form:
rows are tokens, shape $(N, d)$, rows unit-normalized, similarity
$S = W W^{\mathsf T}$.
In particular, `C9-dimensionality-reduction` — the SVD unit — takes
*this very matrix* as its input without reshaping or renaming.
Build the habit now and C9 costs nothing later.

### Checkpoint 3

1. After normalization, what information about each token's raw vector
   has been kept, and what has been thrown away?
2. Why does normalizing twice change nothing? Answer with one algebraic
   observation about `W`'s row norms.
3. Predict the value of `(W[4] * W[4]).sum()` without running, and say
   which word row 4 is.

## 4. Cosine From Unit Rows

For unit vectors the cosine formula collapses: the denominator
$\lVert u \rVert \lVert v \rVert$ equals $1 \cdot 1$, so
$$\cos(u, v) \;=\; u \cdot v
\qquad \text{whenever } \lVert u \rVert = \lVert v \rVert = 1.$$
The normalization *pre-paid* the division.
On rows of `W`, similarity is now a bare dot product.

In [ ]:
cos_hb = (W[0] * W[1]).sum()          # harbor . boat, both unit rows
print(f"cos(harbor, boat) = {cos_hb:.4f}")

# agrees with Session 1's full formula on the raw vectors
u = np.asarray(kv["harbor"], dtype=np.float64)
v = np.asarray(kv["boat"], dtype=np.float64)
full = (u * v).sum() / (np.sqrt((u * u).sum()) * np.sqrt((v * v).sum()))
print(f"full formula      = {full:.4f}")
print("gap:", abs(cos_hb - full))

Same $0.6023$ as Session 1, gap at machine-epsilon scale.
One pair per dot product is still retail arithmetic, though — with $N$
tokens there are $N^2$ ordered pairs, and F3 taught the machine that
computes every one of them in a single expression.

### Checkpoint 4

1. For unit vectors, cosine similarity and the dot product coincide.
   What is the *cheapest* extra fact you also get for free about
   $u \cdot v$'s possible values? (F2 knows.)
2. `W[2]` and `W[3]` are unit rows. Write the shortest expression for
   the cosine of `sailor` and `tide`, and the shape of its result.
3. If someone hands you *raw* rows and you take the bare dot product as
   "similarity", which words get an unfair advantage — long-vector
   words or short-vector words?

## 5. The Similarity Matrix $S = W W^{\mathsf T}$

**The definition, and what each entry is.**
F3 defined the matrix product entrywise:
$(AB)_{ij} = \sum_k A_{ik} B_{kj}$.
Apply it to $A = W$ ($N \times 100$) and $B = W^{\mathsf T}$
($100 \times N$), using $(W^{\mathsf T})_{kj} = W_{jk}$:
$$S_{ij} \;=\; (W W^{\mathsf T})_{ij}
\;=\; \sum_{k} W_{ik}\, W_{jk}
\;=\; w_i \cdot w_j
\;=\; \cos(\text{token } i,\ \text{token } j).$$
Row $i$ dotted with row $j$ — every pairwise cosine, computed at once.
$S$ is $N \times N$: rows and columns are *both* indexed by tokens.
(A matrix of all pairwise dot products of a vector family is called a
**Gram matrix** — $S$ is the Gram matrix of `W`'s rows.)

**Two properties for free, both verifiable in one line each:**

- **Symmetry**: $S_{ij} = w_i \cdot w_j = w_j \cdot w_i = S_{ji}$ —
  the dot product does not care about order.
- **Unit diagonal**: $S_{ii} = w_i \cdot w_i = \lVert w_i \rVert^2 = 1$
  — every token has cosine 1 with itself.

(Practice p11 asks for the entrywise derivation as a graded proof, and
p12 pins down the third property — every entry lies in $[-1, 1]$.)

In [ ]:
S = W @ W.T
print("S shape:", S.shape)

print("symmetry gap :", np.max(np.abs(S - S.T)))
print("diagonal gap :", np.max(np.abs(np.diag(S) - 1.0)))
assert np.allclose(S, S.T) and np.allclose(np.diag(S), 1.0)

np.set_printoptions(precision=3, suppress=True)
print(S)

Read the printout with `WORDS` in hand
(`harbor boat sailor tide violin cello pepper honey`):

- $S_{01} = 0.602$ — `harbor`–`boat`, Session 1's number, now one entry
  among sixty-four;
- $S_{45} = 0.925$ — `violin`–`cello`, the strongest off-diagonal bond
  in the matrix;
- $S_{67} = 0.635$ — `pepper`–`honey`, the pantry cluster;
- $S_{35} = -0.134$ — `tide`–`cello`, the weakest: a mild *negative*
  cosine, directions slightly opposed.

### Checkpoint 5

1. Derive $S_{ij} = w_i \cdot w_j$ from F3's entrywise product formula
   in two lines (this is p11's warm-up).
2. For $N = 30$ tokens: give $S$'s shape, the value of its trace, and
   the number of *distinct* off-diagonal cosines it contains.
3. `S[3].argmax()` returns 3. Is that a bug? What does it tell you, and
   which section of the next lesson deals with it?

## 6. Reading $S$ as a Picture

An $8 \times 8$ grid of numbers hides its structure; a heatmap shows it.
`imshow` maps each entry to a color; tick labels carry the words.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(S, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(WORDS)), WORDS, rotation=45, ha="right")
ax.set_yticks(range(len(WORDS)), WORDS)
fig.colorbar(im, label="cosine similarity")
ax.set_title("S = W @ W.T  (rows and columns are tokens)")
plt.tight_layout()
plt.show()

Three structures leap out:

1. the **diagonal** burns at exactly $1$ — self-similarity;
2. **blocks**: a warm $4 \times 4$ sea block (`harbor boat sailor
   tide`), a hot $2 \times 2$ strings block (`violin cello`), a warm
   pantry pair (`pepper honey`);
3. **cool off-blocks**: sea–strings entries hover near $0$ — the model
   has learned that tides and cellos rarely share a sentence.

This picture *is* stated fact 1 (similar use → nearby vectors) made
visible — and the block structure is exactly what
`C9-dimensionality-reduction` will compress.

### Checkpoint 6

1. Before looking: should the heatmap be symmetric across its main
   diagonal? Why?
2. If `WORDS` were re-ordered alphabetically, which of the three
   structures above survive, and which one gets scrambled?
3. Why is fixing `vmin=-1, vmax=1` the honest choice for a cosine
   heatmap, rather than letting matplotlib auto-scale?

## 7. Worked Exam-Style Example: Numeric Normal-Form MC

Round 1 loves cosine arithmetic small enough for a pencil, wrapped in
the **normal form** device that makes the answer unique-decodable.
Worked in full.

---

**Problem.**
Token vectors $u = (1, 2, 2)$ and $v = (2, 2, 1)$ are rows of a raw
embedding matrix.
Their cosine similarity is a rational number that can be written in
lowest terms as $p/q$ with $\gcd(p, q) = 1$ and $q > 0$.
What is $p + q$?

A. $8$  B. $9$  C. $15$  D. $17$  E. $19$

*Reasoning is not required (but we reason anyway).*

---

**Step 1 — the three ingredients.**
$u \cdot v = 1\cdot2 + 2\cdot2 + 2\cdot1 = 8$;
$\lVert u \rVert = \sqrt{1 + 4 + 4} = 3$;
$\lVert v \rVert = \sqrt{4 + 4 + 1} = 3$.

**Step 2 — assemble and reduce.**
$\cos(u, v) = \dfrac{8}{3 \cdot 3} = \dfrac{8}{9}$.
Already in lowest terms: $\gcd(8, 9) = 1$, $q = 9 > 0$.

**Step 3 — decode.**
$p + q = 8 + 9 = 17$ — answer **D**.
The traps are manufactured from real mistakes: A is the bare dot
product $8$ reported as the answer; B is what "skip the normalization"
decodes to ($8 = 8/1 \to 8 + 1 = 9$); E is a dot-product slip
($1\cdot2 + 2\cdot2\cdot2 = 10 \to 10/9 \to 19$).
The normal form's point: only the *fully correct* fraction decodes to
$17$; each canonical mistake decodes elsewhere.

In [ ]:
u = np.array([1.0, 2.0, 2.0])
v = np.array([2.0, 2.0, 1.0])
cos_uv = (u * v).sum() / (np.sqrt((u * u).sum()) * np.sqrt((v * v).sum()))
print("cosine:", cos_uv, "= 8/9 ->  p + q = 17  -> answer D")

### Checkpoint 7

1. Rework the example with $u = (3, 4)$, $v = (6, 8)$: compute the
   cosine, put it in normal form, and decode $p + q$.
2. Why does the normal form demand $\gcd(p, q) = 1$ *and* $q > 0$ —
   what ambiguity would remain if either constraint were dropped?

## 8. Common Pitfalls II

**Pitfall 1 — the missing `keepdims` (loud version).**
Dividing $(8, 100)$ by $(8,)$: broadcasting aligns *trailing* axes, so
NumPy tries to match $100$ against $8$ and refuses.
The error message is your friend; read the two shapes in it.

In [ ]:
flat_norms = np.sqrt((V * V).sum(axis=1))     # (8,) -- keepdims forgotten
try:
    V / flat_norms
except ValueError as e:
    print("ValueError:", e)

**Pitfall 2 — normalizing the wrong axis (quiet version).**
`axis=0` sums down the *columns*: shape $(1, 100)$, which broadcasts
without complaint — and normalizes nothing you wanted.
No error, wrong numbers: the classic silent shape bug.

In [ ]:
wrong = V / np.sqrt((V * V).sum(axis=0, keepdims=True))   # (1,100): columns!
print("row norms after the wrong normalization:",
      np.round(np.sqrt((wrong * wrong).sum(axis=1)), 3))   # not 1.0

The rows are *not* unit vectors — any downstream "cosine" built from
`wrong` is silently garbage.
The habit that catches it: **assert unit row norms right after
normalizing**, every time, like Section 3's cell did.

**Pitfall 3 — raw dots as similarity.**
Skipping normalization rewards sheer vector length.
The demonstration: scale one vector and watch the two measures react.

In [ ]:
u = V[0]                        # harbor, raw
v = V[1]                        # boat, raw
v3 = 3.0 * v                    # same direction, triple the length

raw_dot = (u * v).sum()
raw_dot3 = (u * v3).sum()
cos_v = (u * v).sum() / (np.sqrt((u*u).sum()) * np.sqrt((v*v).sum()))
cos_v3 = (u * v3).sum() / (np.sqrt((u*u).sum()) * np.sqrt((v3*v3).sum()))
print(f"raw dot: {raw_dot:.3f} -> {raw_dot3:.3f}  (tripled by the rescale)")
print(f"cosine : {cos_v:.4f} -> {cos_v3:.4f}  (unchanged)")

The raw dot triples with the rescale; the cosine does not move.
Since GloVe's raw norms genuinely differ (Section 2's table: $5.07$ to
$6.14$), unnormalized ranking answers "which word is *long*", not
"which word is *close*".

**Pitfall 4 — exact float equality.**
`S[0, 1] == 0.6023` is false — the true value has fifty more digits.
Compare floats with `np.isclose(..., atol=..., rtol=0)` at a stated
tolerance — **`rtol` silently defaults to `1e-5`**, so an `atol=1e-12`
test on a magnitude-5 anchor is really a ~5e-5 test unless you zero it;
this course's anchors say `atol=1e-6` unless a problem states
otherwise.
Exact `==` is reserved for integers, booleans, and shapes.

### Checkpoint 8

1. Classify each pitfall 1–3 as *loud* (raises) or *quiet* (wrong
   numbers, no error), and state the single habit that catches the
   quiet ones.
2. A teammate's `W` passes `W.shape == (8, 100)` yet
   `np.sqrt((W*W).sum(axis=1))` prints values between $3.2$ and
   $3.8$ — like the Pitfall-2 demo's.
   Which pitfall produced this, and how do you know it isn't
   pitfall 1?
3. Write the correct assert for "`S[0, 1]` equals the anchor
   `0.602317` at this course's default tolerance."

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Shape `(12, 100)`. `V[7]` is token 7's full embedding vector,
   shape `(100,)`; `V[:, 7]` is embedding *dimension* 7's value across
   all 12 tokens, shape `(12,)`.
2. "Which word is row $i$?" — the matrix stores only numbers; the
   list is the mapping from row index to word.
3. Transpose it: `V = np.asarray(F, dtype=np.float64).T` (giving
   `(8, 100)`), where `F` is the columns-are-tokens array.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. `(5,)`, `(5, 1)`, and `(1, 3)`.
2. Rows $(3, 4)$ and $(5, 12)$: norms $5$ and $13$; result
   $\begin{pmatrix}5\\13\end{pmatrix}$, shape `(2, 1)`.
3. `np.linalg` (any function) and Python loops; the exam's grading
   note assigns **zero points** to the task when a ban is violated.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Kept: the direction. Thrown away: the length (the raw norm — e.g.
   `boat`'s $6.12$).
2. `W`'s rows already have norm 1, so the second division divides
   every row by $1$ — a no-op.
3. Exactly $1.0$ (up to float error): it is `violin`'s unit row dotted
   with itself, i.e. its squared norm.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. The Cauchy–Schwarz bound: $|u \cdot v| \le \lVert u \rVert \lVert
   v \rVert = 1$, so the dot product of unit vectors always lies in
   $[-1, 1]$ — cosine's range for free (p12 proves it).
2. `(W[2] * W[3]).sum()` — a scalar, shape `()`.
3. Long-vector words: the raw dot product scales linearly with each
   vector's length, so longer vectors post larger dots regardless of
   direction.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. $(W W^{\mathsf T})_{ij} = \sum_k W_{ik}(W^{\mathsf T})_{kj}$
   (entrywise product formula); $(W^{\mathsf T})_{kj} = W_{jk}$, so the
   sum is $\sum_k W_{ik} W_{jk} = w_i \cdot w_j$.
2. Shape `(30, 30)`; trace $= 30$ (thirty ones on the diagonal);
   distinct off-diagonal cosines: $30 \cdot 29 / 2 = 435$ (symmetry
   halves the $870$ ordered pairs).
3. Not a bug: the diagonal entry $S_{33} = 1$ is always the row's
   maximum, i.e. every token's nearest token is itself. Session 3
   (self-exclusion) handles it.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Yes — $S = S^{\mathsf T}$ entry by entry, so the picture must mirror
   across the diagonal.
2. The diagonal survives (it is a property of the *entries*, not the
   order); the visible contiguous *blocks* scramble — block structure
   in a heatmap exists only when related words are adjacent in the
   list. The underlying similarities are unchanged.
3. Cosines live in $[-1, 1]$; fixing the scale maps color to absolute
   meaning (white $\approx 0$, full red $= 1$) and keeps different
   heatmaps comparable. Auto-scaling would stretch the palette over
   the data's accidental range and exaggerate noise.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. $u \cdot v = 18 + 32 = 50$; norms $5$ and $10$;
   $\cos = 50/50 = 1 = 1/1$; $p + q = 2$. (Parallel vectors — $v =
   2u$.)
2. Without $\gcd(p,q) = 1$, the same cosine decodes many ways
   ($8/9 = 16/18 \to 34$); without $q > 0$, sign could be parked in
   the denominator ($-8/-9$) and flip the decode. Together they make
   the representation — and hence $p+q$ — unique.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 1 is loud (ValueError); pitfalls 2 and 3 are quiet. The
   catching habit: assert the invariant you just created — unit row
   norms after normalizing, symmetry/diagonal after building $S$.
2. Pitfall 2 (wrong-axis normalization): the shape contract still
   passes because dividing `(8,100)` by `(1,100)` preserves shape. It
   cannot be pitfall 1 — that one refuses to produce a `W` at all.
3. `assert np.isclose(S[0, 1], 0.602317, atol=1e-6, rtol=0)`

</details>